# Predicting Chronic Kidney Disease

- Kidney disease occurs when the kidneys become damaged and cannot function properly. Chronic kidney disease (CKD) is a major global health concern, affecting approximately 788 million adults worldwide in 2023.

- Early detection is important because kidney disease may progress without noticeable symptoms. This project uses clinical data to predict five target categories: No Disease, Low Risk, Moderate Risk, High Risk, and Severe Disease. We will apply clustering, machine learning algorithms, and a neural network to explore patterns and evaluate prediction performance.

## 1. Loading Dataset

## Dataset Overview

The kidney disease dataset contains **20,538 records and 43 columns**, including 42 input features and one target column. It includes numerical clinical measurements, such as age, blood pressure, blood urea, serum creatinine and haemoglobin, alongside categorical information about diabetes, hypertension, smoking status and other health indicators.

The target column contains five classes: `No_Disease`, `Low_Risk`, `Moderate_Risk`, `High_Risk` and `Severe_Disease`. This dataset will be used to explore patterns, perform clustering, select relevant features, and train machine learning and neural network models to predict the target class.

**Source:** Kaggle – Kidney Disease Dataset  
**Dataset URL:** https://www.kaggle.com/datasets/amanik000/kidney-disease-dataset

In [153]:
import pandas as pd

df = pd.read_csv(
    "../../../assets/datasets/2_KG_Kidney_Disease_Dataset/data/kidney_disease_dataset.csv"
)

df.head()

,Age of the patient,Blood pressure (mm/Hg),Specific gravity of urine,Albumin in urine,Sugar in urine,Red blood cells in urine,Pus cells in urine,Pus cell clumps in urine,Bacteria in urine,Random blood glucose level (mg/dl),...,Smoking status,Body Mass Index (BMI),Physical activity level,Duration of diabetes mellitus (years),Duration of hypertension (years),Cystatin C level,Urinary sediment microscopy results,C-reactive protein (CRP) level,Interleukin-6 (IL-6) level,Target
0,54,167,1.023,1,4,normal,abnormal,not present,not present,96,...,yes,25.3,low,4,16,0.67,normal,4.88,10.23,No_Disease
1,42,127,1.023,3,2,normal,normal,not present,present,73,...,no,20.6,moderate,3,13,0.55,abnormal,4.49,13.11,Low_Risk
2,38,148,1.016,0,0,abnormal,normal,not present,not present,77,...,no,38.4,high,11,23,2.37,abnormal,4.57,13.27,No_Disease
3,7,98,1.017,4,0,abnormal,normal,not present,present,225,...,no,24.7,high,24,3,2.54,abnormal,8.57,12.36,No_Disease
4,67,174,1.015,1,1,normal,abnormal,not present,not present,376,...,yes,17.6,high,22,24,1.90,normal,6.75,1.46,No_Disease


In [154]:
df.isnull().sum()

Age of the patient                             0
Blood pressure (mm/Hg)                         0
Specific gravity of urine                      0
Albumin in urine                               0
Sugar in urine                                 0
Red blood cells in urine                       0
Pus cells in urine                             0
Pus cell clumps in urine                       0
Bacteria in urine                              0
Random blood glucose level (mg/dl)             0
Blood urea (mg/dl)                             0
Serum creatinine (mg/dl)                       0
Sodium level (mEq/L)                           0
Potassium level (mEq/L)                        0
Hemoglobin level (gms)                         0
Packed cell volume (%)                         0
White blood cell count (cells/cumm)            0
Red blood cell count (millions/cumm)           0
Hypertension (yes/no)                          0
Diabetes mellitus (yes/no)                     0
Coronary artery dise

**Observation**

The dataset contains no missing values across all 43 columns. Therefore, no missing-value imputation is required during preprocessing.

In [155]:
df.duplicated().sum()

np.int64(0)

**Observation**

The dataset contains 0 duplicate records. Therefore, no duplicate rows need to be removed.

## 2. Data Preprocessing

In [156]:
df.select_dtypes(include='object').nunique()

Red blood cells in urine                    2
Pus cells in urine                          2
Pus cell clumps in urine                    2
Bacteria in urine                           2
Hypertension (yes/no)                       2
Diabetes mellitus (yes/no)                  2
Coronary artery disease (yes/no)            2
Appetite (good/poor)                        2
Pedal edema (yes/no)                        2
Anemia (yes/no)                             2
Family history of chronic kidney disease    2
Smoking status                              2
Physical activity level                     3
Urinary sediment microscopy results         2
Target                                      5
dtype: int64

**Observation**

The dataset contains 15 categorical variables. Most are binary variables, while Physical activity level contains three categories and Target contains five classes. These categorical variables will need to be encoded before model training.

### 2.1 Encoding Categorical Variables

- Categorical variables are converted into numerical representations so that they can be used by machine learning algorithms.

In [157]:
df.select_dtypes(include='object').apply(lambda x: x.unique())

Red blood cells in urine                                                   [normal, abnormal]
Pus cells in urine                                                         [abnormal, normal]
Pus cell clumps in urine                                               [not present, present]
Bacteria in urine                                                      [not present, present]
Hypertension (yes/no)                                                               [yes, no]
Diabetes mellitus (yes/no)                                                          [yes, no]
Coronary artery disease (yes/no)                                                    [no, yes]
Appetite (good/poor)                                                             [good, poor]
Pedal edema (yes/no)                                                                [no, yes]
Anemia (yes/no)                                                                     [no, yes]
Family history of chronic kidney disease                    

**Observation**

The output shows that the dataset contains categorical features with either two or more than two unique values. Therefore, encoding will be performed to convert categorical data into a numerical format suitable for machine learning algorithms.

The following encoding techniques will be used:

- **Binary Encoding:** Features with two unique categories will be converted into numerical values (0 and 1).
- **One-Hot Encoding:** Features with more than two categories, such as *Physical activity level*, will be converted into separate binary columns for each category.
- **Label Encoding:** The `Target` column will be converted into numerical labels representing the five kidney disease classes.

These encoding techniques will make the dataset suitable for subsequent feature engineering, clustering and model training.

In [158]:
df_processed = df.copy()

### Binary Categorical Encoding

The binary categorical variables are converted into numerical values, where the two categories are represented by 0 and 1.

In [159]:
binary_columns = [
    'Red blood cells in urine',
    'Pus cells in urine',
    'Pus cell clumps in urine',
    'Bacteria in urine',
    'Hypertension (yes/no)',
    'Diabetes mellitus (yes/no)',
    'Coronary artery disease (yes/no)',
    'Appetite (good/poor)',
    'Pedal edema (yes/no)',
    'Anemia (yes/no)',
    'Family history of chronic kidney disease',
    'Smoking status',
    'Urinary sediment microscopy results'
]

for col in binary_columns:
    df_processed[col] = df_processed[col].astype('category').cat.codes

In [160]:
df_processed[binary_columns].head()

,Red blood cells in urine,Pus cells in urine,Pus cell clumps in urine,Bacteria in urine,Hypertension (yes/no),Diabetes mellitus (yes/no),Coronary artery disease (yes/no),Appetite (good/poor),Pedal edema (yes/no),Anemia (yes/no),Family history of chronic kidney disease,Smoking status,Urinary sediment microscopy results
0,1,0,0,0,1,1,0,0,0,0,0,1,1
1,1,1,0,1,0,1,0,0,1,1,1,0,0
2,0,1,0,0,0,0,1,0,1,0,0,0,0
3,0,1,0,1,0,0,1,0,0,1,0,0,0
4,1,0,0,0,0,0,0,0,1,1,0,1,1


### Encoding Physical Activity Level

The `Physical activity level` variable contains three categories: low, moderate, and high. One-hot encoding is used to represent these categories numerically without assuming an order between them.

In [161]:
df_processed = pd.get_dummies(
    df_processed,
    columns=['Physical activity level'],
    dtype=int
)

In [162]:
df_processed.head()

,Age of the patient,Blood pressure (mm/Hg),Specific gravity of urine,Albumin in urine,Sugar in urine,Red blood cells in urine,Pus cells in urine,Pus cell clumps in urine,Bacteria in urine,Random blood glucose level (mg/dl),...,Duration of diabetes mellitus (years),Duration of hypertension (years),Cystatin C level,Urinary sediment microscopy results,C-reactive protein (CRP) level,Interleukin-6 (IL-6) level,Target,Physical activity level_high,Physical activity level_low,Physical activity level_moderate
0,54,167,1.023,1,4,1,0,0,0,96,...,4,16,0.67,1,4.88,10.23,No_Disease,0,1,0
1,42,127,1.023,3,2,1,1,0,1,73,...,3,13,0.55,0,4.49,13.11,Low_Risk,0,0,1
2,38,148,1.016,0,0,0,1,0,0,77,...,11,23,2.37,0,4.57,13.27,No_Disease,1,0,0
3,7,98,1.017,4,0,0,1,0,1,225,...,24,3,2.54,0,8.57,12.36,No_Disease,1,0,0
4,67,174,1.015,1,1,1,0,0,0,376,...,22,24,1.90,1,6.75,1.46,No_Disease,1,0,0


### Encoding the Target Variable

The target variable contains five kidney disease risk classes. These categorical labels are converted into numerical values so that they can be used as the output variable during model training.

In [163]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

df_processed['Target'] = label_encoder.fit_transform(df_processed['Target'])

In [164]:
print(dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_))))

{'High_Risk': np.int64(0), 'Low_Risk': np.int64(1), 'Moderate_Risk': np.int64(2), 'No_Disease': np.int64(3), 'Severe_Disease': np.int64(4)}


**Observation**

The five target classes were successfully converted into numerical labels using LabelEncoder. Each kidney disease risk category is now represented by a unique numerical value, which can be used as the target variable during model training.

In [165]:
df_processed.head()

,Age of the patient,Blood pressure (mm/Hg),Specific gravity of urine,Albumin in urine,Sugar in urine,Red blood cells in urine,Pus cells in urine,Pus cell clumps in urine,Bacteria in urine,Random blood glucose level (mg/dl),...,Duration of diabetes mellitus (years),Duration of hypertension (years),Cystatin C level,Urinary sediment microscopy results,C-reactive protein (CRP) level,Interleukin-6 (IL-6) level,Target,Physical activity level_high,Physical activity level_low,Physical activity level_moderate
0,54,167,1.023,1,4,1,0,0,0,96,...,4,16,0.67,1,4.88,10.23,3,0,1,0
1,42,127,1.023,3,2,1,1,0,1,73,...,3,13,0.55,0,4.49,13.11,1,0,0,1
2,38,148,1.016,0,0,0,1,0,0,77,...,11,23,2.37,0,4.57,13.27,3,1,0,0
3,7,98,1.017,4,0,0,1,0,1,225,...,24,3,2.54,0,8.57,12.36,3,1,0,0
4,67,174,1.015,1,1,1,0,0,0,376,...,22,24,1.90,1,6.75,1.46,3,1,0,0


**Observation**

After preprocessing, all categorical variables have been converted into numerical representations. The three categories of `Physical activity level` were represented using one-hot encoding, resulting in three additional columns. The dataset now contains 45 columns with no missing values.

## 3. Feature Engineering

Feature engineering involves creating meaningful new features from the existing variables to potentially improve the performance of machine learning models.

In [166]:
# Correlation of numerical features with the target
correlations = df_processed.corr(numeric_only=True)['Target'].sort_values(ascending=False)

correlations

Target                                         1.000000
Pus cells in urine                             0.026725
Specific gravity of urine                      0.014126
Pedal edema (yes/no)                           0.012895
Age of the patient                             0.011191
Red blood cells in urine                       0.009634
Hypertension (yes/no)                          0.009548
Duration of diabetes mellitus (years)          0.008891
Parathyroid hormone (PTH) level                0.007211
Diabetes mellitus (yes/no)                     0.006088
Hemoglobin level (gms)                         0.005820
C-reactive protein (CRP) level                 0.004926
Random blood glucose level (mg/dl)             0.004437
Physical activity level_high                   0.003954
Serum albumin level                            0.003526
Serum calcium level                            0.002980
Blood urea (mg/dl)                             0.002709
Physical activity level_moderate               0

**Observation**

The correlation values between the numerical features and the encoded target are generally close to zero. However, because the target represents five categorical classes encoded as numerical labels, these correlations should not be used alone to determine feature importance. Further feature evaluation will be performed during the modelling and feature selection stages.

### 3.1 Creating a New Feature – Diabetes and Hypertension Interaction

- Diabetes and hypertension are both important health conditions associated with chronic kidney disease. The dataset contains separate variables for these two conditions, but their combined presence may provide additional information for the prediction task.

- Therefore, a new binary feature is created to indicate whether a patient has both diabetes and hypertension. This allows the model to directly represent their combined occurrence rather than relying only on the two individual features.

- The usefulness of this interaction feature will be evaluated during feature selection and model training.

**Reference:** Zhang et al. (2019), "Synergistic interaction of hypertension and diabetes on chronic kidney disease: Insights from the National Health and Nutrition Examination Survey 1999-2006."

In [167]:
df_processed['Diabetes_Hypertension_Interaction'] = (
    df_processed['Diabetes mellitus (yes/no)'] *
    df_processed['Hypertension (yes/no)']
)

In [168]:
df_processed['Diabetes_Hypertension_Interaction'].value_counts()

Diabetes_Hypertension_Interaction
0    15477
1     5061
Name: count, dtype: int64

**Observation**

The `Diabetes_Hypertension_Interaction` feature contains 15,477 records with a value of 0 and 5,061 records with a value of 1.

A value of 1 indicates that both diabetes and hypertension are present, while 0 indicates that both conditions are not present simultaneously. Approximately 24.65% of the records have both conditions.

This feature will be retained as a candidate feature and its contribution to the prediction task will be evaluated during feature selection and model training.

### 3.2 Creating a New Feature – BMI Age Interaction

The `BMI_Age_Interaction` feature is created by multiplying the patient's Body Mass Index (BMI) by their age.

**Formula:**

`BMI_Age_Interaction = BMI × Age`

**Why was this feature created?**

Age and BMI are both potentially relevant factors in kidney disease prediction. This interaction feature was introduced to explore whether their combined effect could provide additional information that might not be captured by considering each feature independently.

The original Age and BMI features are retained, allowing us to evaluate whether the interaction provides additional predictive value during feature selection and model training.

In [169]:

# Create a candidate BMI-age interaction feature
df_processed["BMI_Age_Interaction"] = (
    df_processed["Body Mass Index (BMI)"]
    * df_processed["Age of the patient"]
)

print(df_processed["BMI_Age_Interaction"].describe())



count    20538.000000
mean      1305.137662
std        785.091469
min         75.500000
25%        659.700000
50%       1208.200000
75%       1826.150000
max       3600.000000
Name: BMI_Age_Interaction, dtype: float64


In [170]:
display(
    df_processed.groupby("Target")["BMI_Age_Interaction"]
    .agg(["count", "mean", "median", "std"])
    .round(3)
)

,count,mean,median,std
Target,,,,
0,821,1255.655,1172.0,761.827
1,2054,1297.655,1225.0,785.183
2,821,1278.674,1179.2,767.765
3,16432,1310.495,1211.7,787.581
4,410,1279.988,1182.6,762.957


In [171]:
from scipy.stats import kruskal

groups = [
    group["BMI_Age_Interaction"].values
    for _, group in df_processed.groupby("Target", observed=True)
]

statistic, p_value = kruskal(*groups)

print(f"Kruskal-Wallis statistic: {statistic:.3f}")
print(f"P-value: {p_value:.6g}")

Kruskal-Wallis statistic: 4.304
P-value: 0.366453


**Observations**

The `BMI_Age_Interaction` feature was evaluated across the five target classes. The Kruskal–Wallis test produced a statistic of **4.304** and a p-value of **0.366453**. The feature will be retained as an experimental candidate for further evaluation during feature selection and model training.

## 4. Saving the Processed Dataset

In [174]:
df_processed.to_csv(
    "../../../assets/datasets/2_KG_Kidney_Disease_Dataset/data/kidney_disease_dataset_processed.csv",
    index=False
)

In [175]:
# df_processed.shape